In [ ]:
# ===== NOTEBOOK CELL 1: SETUP =====
import os
import torch
from pathlib import Path

KAGGLE_DATA = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "train.csv" in files and "competitions" in root and "rsna-knee-abnormality-detection" in root:
        KAGGLE_DATA = Path(root)
        break

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"KAGGLE_DATA: {KAGGLE_DATA}")

In [ ]:
# ===== NOTEBOOK CELL 2: INSTALL MONAI (offline-safe) =====
import os
try:
    import monai
    print(f"MONAI already available: {monai.__version__}")
except ImportError:
    wheel_dir = None
    for root, dirs, files in os.walk("/kaggle/input"):
        if any(f.startswith("monai") and f.endswith(".whl") for f in files):
            wheel_dir = root
            break
    if wheel_dir:
        print(f"Installing MONAI from offline wheel in: {wheel_dir}")
        !pip install --no-index --find-links={wheel_dir} monai --quiet
    else:
        print("No offline MONAI wheel found — attempting live install (requires internet).")
        !pip install monai --quiet
    import monai
    print(f"MONAI installed: {monai.__version__}")

In [ ]:
# ===== NOTEBOOK CELL 3: LOAD LABELS + SERIES INDEX + REPORT TEXT =====
import pandas as pd
from pathlib import Path

labels_df = pd.read_csv("/kaggle/input/datasets/stacylisatoriola/rsna-knee-model-best/train_labels_filled.csv")
target_columns = [c for c in labels_df.columns if c != "StudyInstanceUID"]

train_translated = pd.read_csv("/kaggle/input/datasets/stacylisatoriola/rsna-knee-model-best/train_translated.csv")
labels_df = labels_df.merge(train_translated[["StudyInstanceUID", "Report_English"]], on="StudyInstanceUID", how="left")
labels_df["Report_English"] = labels_df["Report_English"].fillna("")

train_series = pd.read_csv(KAGGLE_DATA / "train_series.csv")
series_root = KAGGLE_DATA / "train_series"

print(f"Labels: {len(labels_df)} studies, {len(target_columns)} targets")
print(f"Reports attached: {(labels_df['Report_English'].str.len() > 0).sum()} non-empty")

In [ ]:
# ===== NOTEBOOK CELL 4: 3D VOLUME LOADER (subsampled + cached + augmentation) =====
import os, random, numpy as np, torch, cv2, pydicom
import torch.nn.functional as F

TARGET_SIZE = 160
FIXED_DEPTH = 32

CACHE_DIR = "/kaggle/working/volume_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

def get_real_file_list(study_uid, series_root, train_series_df):
    study_series = train_series_df[train_series_df["StudyInstanceUID"] == study_uid]
    all_files = []
    for _, row in study_series.iterrows():
        series_dir = series_root / study_uid / row["SeriesInstanceUID"]
        for fname in os.listdir(series_dir):
            all_files.append(series_dir / fname)
    if not all_files:
        raise ValueError(f"No real files found for study {study_uid}")
    return sorted(all_files)

def load_study_volume(study_uid, series_root, train_series_df):
    all_files = get_real_file_list(study_uid, series_root, train_series_df)
    if len(all_files) > FIXED_DEPTH:
        indices = np.linspace(0, len(all_files) - 1, FIXED_DEPTH).round().astype(int)
        all_files = [all_files[i] for i in sorted(set(indices))]
    slices = []
    for dcm_path in all_files:
        try:
            dcm = pydicom.dcmread(dcm_path)
            arr = dcm.pixel_array.astype(np.float32)
            arr = cv2.resize(arr, (TARGET_SIZE, TARGET_SIZE))
            lo, hi = arr.min(), arr.max()
            if hi > lo:
                arr = (arr - lo) / (hi - lo)
            slices.append(arr)
        except Exception as e:
            print(f"Skipped unreadable file {dcm_path}: {e}")
    if not slices:
        raise ValueError(f"No real slices loaded for study {study_uid}")
    vol = np.stack(slices, axis=0)
    vol = torch.from_numpy(vol).float()
    vol = vol.unsqueeze(0).unsqueeze(0)
    vol = F.interpolate(vol, size=(FIXED_DEPTH, TARGET_SIZE, TARGET_SIZE), mode="trilinear", align_corners=False)
    return vol.squeeze(0)

def load_study_volume_cached(study_uid, series_root, train_series_df):
    cache_path = os.path.join(CACHE_DIR, f"{study_uid}.pt")
    if os.path.exists(cache_path):
        return torch.load(cache_path)
    vol = load_study_volume(study_uid, series_root, train_series_df)
    torch.save(vol, cache_path)
    return vol

def transform_volume(vol):
    # vol shape: (1, D, H, W). Call this only on training data, never validation/test.
    if random.random() < 0.5:
        vol = torch.flip(vol, dims=[3])  # left-right flip
    if random.random() < 0.5:
        vol = torch.flip(vol, dims=[2])  # up-down flip
    if random.random() < 0.5:
        k = random.choice([1, 2, 3])
        vol = torch.rot90(vol, k=k, dims=[2, 3])  # in-plane rotation
    if random.random() < 0.5:
        scale = 1.0 + (random.random() - 0.5) * 0.2   # ±10% intensity scale
        shift = (random.random() - 0.5) * 0.1          # ±5% intensity shift
        vol = torch.clamp(vol * scale + shift, 0.0, 1.0)
    if random.random() < 0.3:
        noise = torch.randn_like(vol) * 0.02
        vol = torch.clamp(vol + noise, 0.0, 1.0)
    return vol

In [ ]:
# ===== NOTEBOOK CELL 5: MONAI 3D MODEL + TEXT FUSION (MedicalNet pretrained backbone, offline-safe) =====
import os
import torch.nn as nn
from monai.networks.nets import resnet18

class TextEncoder(nn.Module):
    def __init__(self, vocab_size, out_dim=64):
        super().__init__()
        self.proj = nn.Linear(vocab_size, out_dim)
    def forward(self, tfidf):
        return self.proj(tfidf)

def build_image_backbone():
    # Build the architecture untrained first (pretrained=False avoids any live download attempt).
    backbone = resnet18(
        pretrained=False,
        spatial_dims=3,
        n_input_channels=1,
        feed_forward=False,
        shortcut_type="A",
    )
    # Then load the MedicalNet weights we staged offline as a Kaggle dataset.
    weights_path = None
    for root, dirs, files in os.walk("/kaggle/input"):
        if "medicalnet_resnet18_23dataset.pth" in files:
            weights_path = os.path.join(root, "medicalnet_resnet18_23dataset.pth")
            break
    if weights_path:
        state_dict = torch.load(weights_path, map_location=device)
        backbone.load_state_dict(state_dict, strict=True)
        print(f"Loaded MedicalNet pretrained weights from: {weights_path}")
    else:
        print("MedicalNet weights dataset not found as an attached Input — using random initialization instead.")
    return backbone

class Knee3DModel(nn.Module):
    def __init__(self, num_targets, vocab_size, text_feat_dim=64):
        super().__init__()
        self.backbone = build_image_backbone()
        self.text_encoder = TextEncoder(vocab_size, out_dim=text_feat_dim)
        self.classifier = nn.Linear(512 + text_feat_dim, num_targets)

    def forward(self, x, text_tfidf):
        img_feat = self.backbone(x)
        text_feat = self.text_encoder(text_tfidf)
        fused = torch.cat([img_feat, text_feat], dim=1)
        return self.classifier(fused)

print("Knee3DModel defined — image backbone initializes with offline-staged MedicalNet pretrained weights when the dataset is attached.")

In [ ]:
# ===== NOTEBOOK CELL 6: TRAIN 3D MODEL — IMAGE-FOCUSED + RESUMABLE (v4) =====
import time
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score

train_studies, val_studies = train_test_split(labels_df, test_size=0.10, random_state=0)

def fit_tfidf_vectorizer(report_texts, max_features=300):
    v = TfidfVectorizer(max_features=max_features)
    v.fit(report_texts)
    return v

vectorizer = fit_tfidf_vectorizer(train_studies["Report_English"].tolist(), max_features=300)
vocab_size = len(vectorizer.vocabulary_)
print(f"TF-IDF vocab size: {vocab_size}")

import joblib
joblib.dump(vectorizer, "/kaggle/working/tfidf_vectorizer.joblib")
print("Saved vectorizer.")

class KneeStudyDataset(Dataset):
    def __init__(self, df, series_root, train_series_df, vectorizer, target_columns, transform=False):
        self.df = df.reset_index(drop=True)
        self.series_root = series_root
        self.train_series_df = train_series_df
        self.vectorizer = vectorizer
        self.target_columns = target_columns
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            vol = load_study_volume_cached(row["StudyInstanceUID"], self.series_root, self.train_series_df)
        except Exception:
            vol = torch.zeros(1, FIXED_DEPTH, TARGET_SIZE, TARGET_SIZE)
        if self.transform:
            vol = transform_volume(vol)
        labels = torch.tensor(row[self.target_columns].values.astype(float), dtype=torch.float32)
        text_vec = self.vectorizer.transform([row["Report_English"]]).toarray()[0]
        text_tfidf = torch.tensor(text_vec, dtype=torch.float32)
        return vol, text_tfidf, labels

BATCH_SIZE = 8
NUM_WORKERS = 4
TEXT_DROPOUT_PROB = 0.85

train_dataset = KneeStudyDataset(train_studies, series_root, train_series, vectorizer, target_columns, transform=True)
val_dataset = KneeStudyDataset(val_studies, series_root, train_series, vectorizer, target_columns, transform=False)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

n = len(labels_df)
pos_weight_values = [(n - int(labels_df[c].sum())) / int(labels_df[c].sum()) if labels_df[c].sum() > 0 else 1.0 for c in target_columns]
pos_weight_tensor = torch.tensor(pos_weight_values, dtype=torch.float32).to(device)

# Reverted: per-target task weighting removed — it regressed the public score (0.535 -> 0.510).
# Back to plain pos_weight-balanced loss, no extra per-target multiplier.
loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

model = Knee3DModel(num_targets=len(target_columns), vocab_size=vocab_size).to(device)

# NEW: resume from a previous best checkpoint instead of starting fresh each run, if one is attached.
resume_path = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "model_3d_best.pt" in files:
        resume_path = os.path.join(root, "model_3d_best.pt")
        break
if resume_path:
    model.load_state_dict(torch.load(resume_path, map_location=device))
    print(f"Resumed training from previous checkpoint: {resume_path}")
else:
    print("No previous checkpoint found as an Input — starting fresh (MedicalNet-pretrained backbone).")

MAX_EPOCHS = 50
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
scaler = torch.amp.GradScaler("cuda")

START = time.time()
BUDGET_S = 8 * 3600
PATIENCE = 6
epochs_without_improvement = 0
best_auc = -1.0

for epoch in range(MAX_EPOCHS):
    model.train()
    losses = []
    t0 = time.time()
    for vol, text_tfidf, labels in train_loader:
        vol, text_tfidf, labels = vol.to(device), text_tfidf.to(device), labels.to(device)
        dropout_mask = (torch.rand(text_tfidf.size(0), 1, device=device) > TEXT_DROPOUT_PROB).float()
        text_tfidf = text_tfidf * dropout_mask

        optimizer.zero_grad()
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(vol, text_tfidf)
            loss = loss_fn(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        losses.append(loss.item())

        if time.time() - START > BUDGET_S:
            print(f"\nTime budget ({BUDGET_S/3600:.1f}h) reached mid-epoch — saving and stopping.")
            torch.save(model.state_dict(), "/kaggle/working/model_3d_last.pt")
            break
    scheduler.step()

    print(f"Epoch {epoch+1}/{MAX_EPOCHS} — train loss: {np.mean(losses):.4f} — {round(time.time()-t0)}s "
          f"— elapsed total: {round((time.time()-START)/60)} min — lr: {scheduler.get_last_lr()[0]:.2e}")

    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for vol, text_tfidf, labels in val_loader:
            vol = vol.to(device)
            text_tfidf_zeroed = torch.zeros_like(text_tfidf).to(device)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(vol, text_tfidf_zeroed)
            all_probs.append(torch.sigmoid(logits).float().cpu().numpy())
            all_labels.append(labels.numpy())
    all_probs = np.concatenate(all_probs, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    aucs = []
    for idx, target in enumerate(target_columns):
        try:
            auc = roc_auc_score(all_labels[:, idx], all_probs[:, idx])
        except ValueError:
            auc = float("nan")
        aucs.append(auc)
        print(f"  {target}: AUC = {auc:.4f}")
    avg_auc = np.nanmean(aucs)
    print(f"  Average AUC (text-free, honest): {avg_auc:.4f}")

    torch.save(model.state_dict(), f"/kaggle/working/model_3d_epoch{epoch+1}.pt")
    if avg_auc > best_auc:
        best_auc = avg_auc
        epochs_without_improvement = 0
        torch.save(model.state_dict(), "/kaggle/working/model_3d_best.pt")
        print(f"  New best model saved (epoch {epoch+1}, AUC {avg_auc:.4f})")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f"\nEarly stopping — no improvement for {PATIENCE} epochs. Best AUC: {best_auc:.4f}")
            break

    if time.time() - START > BUDGET_S:
        break

print(f"\nDone. Best AUC: {best_auc:.4f}. Total time: {round((time.time()-START)/60)} min")

In [ ]:
# ===== NOTEBOOK CELL 7: FINAL PREDICTION — model/vectorizer lookup (handles both cases) =====
import os, numpy as np, pandas as pd, torch, joblib

def find_file(filename):
    # Check /kaggle/working first (same-session commit), then /kaggle/input (recovered from a prior version's Output)
    working_path = f"/kaggle/working/{filename}"
    if os.path.exists(working_path):
        return working_path
    for root, dirs, files in os.walk("/kaggle/input"):
        if filename in files:
            return os.path.join(root, filename)
    return None

model_weights_path = find_file("model_3d_best.pt")
vectorizer_path = find_file("tfidf_vectorizer.joblib")
if model_weights_path is None or vectorizer_path is None:
    raise FileNotFoundError("Could not find model_3d_best.pt or tfidf_vectorizer.joblib in /kaggle/working or /kaggle/input.")

print(f"Using model weights: {model_weights_path}")
print(f"Using vectorizer: {vectorizer_path}")

vectorizer = joblib.load(vectorizer_path)
vocab_size = len(vectorizer.vocabulary_)

model = Knee3DModel(num_targets=len(target_columns), vocab_size=vocab_size).to(device)
model.load_state_dict(torch.load(model_weights_path, map_location=device))
model.eval()

test_df = pd.read_csv(KAGGLE_DATA / "test.csv")
test_series = pd.read_csv(KAGGLE_DATA / "test_series.csv")
test_series_root = KAGGLE_DATA / "test_series"
sample_submission = pd.read_csv(KAGGLE_DATA / "sample_submission.csv")
official_target_order = [c for c in sample_submission.columns if c != "StudyInstanceUID"]

empty_text_vec = vectorizer.transform([""]).toarray()
empty_text_tfidf = torch.tensor(empty_text_vec, dtype=torch.float32).to(device)

predictions = []
with torch.no_grad():
    for _, row in test_df.iterrows():
        study_uid = row["StudyInstanceUID"]
        try:
            vol = load_study_volume(study_uid, test_series_root, test_series).unsqueeze(0).to(device)
            if device.type == "cuda":
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    logits = model(vol, empty_text_tfidf)
            else:
                logits = model(vol, empty_text_tfidf)
            probs = torch.sigmoid(logits).float().cpu().numpy()[0]
        except Exception as e:
            print(f"Skipped {study_uid}: {e}")
            probs = np.full(len(target_columns), 0.5)
        predictions.append([study_uid] + probs.tolist())

submission_df = pd.DataFrame(predictions, columns=["StudyInstanceUID"] + target_columns)
submission_df = submission_df[["StudyInstanceUID"] + official_target_order]
submission_df.to_csv("/kaggle/working/submission.csv", index=False)
print("Saved submission.csv")
print(submission_df.head())